In [ ]:
from PIL import Image
import os
import numpy as np
import random
from glob import glob
from tqdm import tqdm
import shutil
from scipy.io import loadmat, savemat 
import csv
import matplotlib.pyplot as plt
import json
from copy import deepcopy
def create_dir(directory):
    if not os.path.exists(directory):
        os.makedirs(directory)


In [ ]:
slide_path='../../data/Leica/Leica_aivis/'
json_list=glob('../../data/Leica/Leica_aivis/results/*.json')
slide_list=[]
pair_json_list=[]
for i in tqdm(range(len(json_list))):
    with open(json_list[i], 'r') as f:
        data = json.load(f)
    slide_name=data['images'][0]['file_name']
    slide_name=slide_name.split('_')[0]+';'+slide_name.split('_')[1]+';'+slide_name.split('_')[2]+';'+slide_name.split('_')[3]+'_'+slide_name.split('_')[4]

    if os.path.exists(slide_path+slide_name):
        slide_list.append(slide_path+slide_name)
        pair_json_list.append(json_list[i])

In [ ]:
her2_json_list=[]
er_pr_json_list=[]
ki67_json_list=[]
her2_slide_list=[]
er_pr_slide_list=[]
ki67_slide_list=[]
for i in tqdm(range(len(pair_json_list))):
    json_name=pair_json_list[i].split('/')[-1]
    if '_CerbB' in json_name:
        her2_json_list.append(pair_json_list[i])
        her2_slide_list.append(slide_list[i])
    elif '_ER' in json_name:
        er_pr_json_list.append(pair_json_list[i])
        er_pr_slide_list.append(slide_list[i])
    elif '_PR' in json_name:
        er_pr_json_list.append(pair_json_list[i])
        er_pr_slide_list.append(slide_list[i])

    elif '_Ki-67' in json_name:
        ki67_json_list.append(pair_json_list[i])
        ki67_slide_list.append(slide_list[i])
    else:
        print(json_name)

In [ ]:
#시각화 코드
import matplotlib.pyplot as plt
import openslide
from PIL import Image, ImageCms
import io
import cv2

# ── Aperio ICC 프로파일 적용 ──────────
def get_icc_transform(slide):
    str_icc_size = slide.properties.get('openslide.icc-size')
    if str_icc_size is None or int(str_icc_size) == 0:
        return None
    try:
        bytes_icc = slide.color_profile.tobytes()
    except AttributeError:
        return None
    icc_profile = ImageCms.ImageCmsProfile(io.BytesIO(bytes_icc))
    srgb_profile = ImageCms.createProfile('sRGB')
    transform = ImageCms.buildTransform(
        icc_profile, srgb_profile,
        'RGB', 'RGB',
        renderingIntent=ImageCms.Intent.PERCEPTUAL
    )
    return transform

def apply_icc(np_patch, icc_transform):
    if icc_transform is None:
        return np_patch
    pil_img = Image.fromarray(np_patch)
    pil_img = ImageCms.applyTransform(pil_img, icc_transform)
    return np.array(pil_img)

downsample_factor=20
i=3
slide_path=her2_slide_list[i]
json_path=her2_json_list[i]
slide = openslide.OpenSlide(slide_path)
with open(json_path, 'r') as f:
    data = json.load(f)

icc_transform = get_icc_transform(slide)
print(f"ICC profile: {'있음' if icc_transform is not None else '없음'}")

img = slide.get_thumbnail((slide.level_dimensions[0][0]//downsample_factor,slide.level_dimensions[0][1]//downsample_factor))
img = np.array(img.convert('RGB'))
img = apply_icc(img, icc_transform)
img = Image.fromarray(img)

class_tuple={}
for i in range(len(data['categories'])):
    class_tuple[data['categories'][i]['id']]=data['categories'][i]['name']
    class_tuple[4]='other'

# 전체 슬라이드 시각화 (클래스별 색상)
img_class = img.copy()
for i in range(len(data['annotations'])):
    x=data['annotations'][i]['bbox'][0]
    y=data['annotations'][i]['bbox'][1]
    w=data['annotations'][i]['bbox'][2]
    h=data['annotations'][i]['bbox'][3]
    img_x=int(x/downsample_factor)
    img_y=int(y/downsample_factor)
    if class_tuple[data['annotations'][i]['category_id']]=='class0':
        img_class.putpixel((img_x, img_y), (0,255, 0))
    if class_tuple[data['annotations'][i]['category_id']]=='class1':
        img_class.putpixel((img_x, img_y), (128,128, 0))
    if class_tuple[data['annotations'][i]['category_id']]=='class2':
        img_class.putpixel((img_x, img_y), (0, 0, 255))
    if class_tuple[data['annotations'][i]['category_id']]=='class3':
        img_class.putpixel((img_x, img_y), (255, 0, 255))
    if class_tuple[data['annotations'][i]['category_id']]=='class4':
        img_class.putpixel((img_x, img_y), (255, 0, 0))

# 전체 슬라이드 시각화 (was_nonT 색상: True=초록, False=빨강)
img_nont = img.copy()
for i in range(len(data['annotations'])):
    x=data['annotations'][i]['bbox'][0]
    y=data['annotations'][i]['bbox'][1]
    img_x=int(x/downsample_factor)
    img_y=int(y/downsample_factor)
    if data['annotations'][i].get('was_nonT', False):
        img_nont.putpixel((img_x, img_y), (0, 255, 0))
    else:
        img_nont.putpixel((img_x, img_y), (255, 0, 0))

fig, axes = plt.subplots(1, 3, figsize=(60, 20))
axes[0].imshow(img)
axes[0].set_title('Original slide (ICC)')
axes[1].imshow(img_class)
axes[1].set_title('Class-based coloring')
axes[2].imshow(img_nont)
axes[2].set_title('was_nonT (Green=True, Red=False)')
plt.tight_layout()
plt.show()

In [ ]:
# 랜덤 위치 5000x5000 crop 시각화
from PIL import ImageDraw
crop_size = 5000
slide_w, slide_h = slide.level_dimensions[0]
rand_x = random.randint(0, max(0, 80000))
rand_y = random.randint(0, max(0, slide_h - crop_size))
print(f"Random crop position: ({rand_x}, {rand_y})")

crop_img = slide.read_region((rand_x, rand_y), 0, (crop_size, crop_size)).convert('RGB')
crop_img = np.array(crop_img)
crop_img = apply_icc(crop_img, icc_transform)
crop_img = Image.fromarray(crop_img)

crop_img_class = crop_img.copy()
crop_img_nont = crop_img.copy()
draw_class = ImageDraw.Draw(crop_img_class)
draw_nont = ImageDraw.Draw(crop_img_nont)

for i in range(len(data['annotations'])):
    x = data['annotations'][i]['bbox'][0]
    y = data['annotations'][i]['bbox'][1]
    w = data['annotations'][i]['bbox'][2]
    h = data['annotations'][i]['bbox'][3]
    # crop 범위 내의 annotation만 표시
    if rand_x <= x < rand_x + crop_size and rand_y <= y < rand_y + crop_size:
        cx = int(x - rand_x)
        cy = int(y - rand_y)
        cw = int(w)
        ch = int(h)
        # 클래스별 색상
        cat_name = class_tuple[data['annotations'][i]['category_id']]
        if cat_name == 'class0':
            color = (0, 255, 0)
        elif cat_name == 'class1':
            color = (128, 128, 0)
        elif cat_name == 'class2':
            color = (0, 0, 255)
        elif cat_name == 'class3':
            color = (255, 0, 255)
        elif cat_name == 'other':
            color = (255, 0, 0)
        else:
            color = (255, 255, 255)
        draw_class.rectangle([cx, cy, cx + cw, cy + ch], outline=color, width=2)

        # was_nonT 색상
        if data['annotations'][i].get('was_nonT', False):
            nont_color = (0, 255, 0)
        else:
            nont_color = (255, 0, 0)
        draw_nont.rectangle([cx, cy, cx + cw, cy + ch], outline=nont_color, width=2)

fig, axes = plt.subplots(3, 1, figsize=(20, 60))
axes[0].imshow(crop_img)
axes[0].set_title(f'Original ICC ({rand_x}, {rand_y})')
axes[1].imshow(crop_img_class)
axes[1].set_title(f'Crop {crop_size}x{crop_size} - Class-based ({rand_x}, {rand_y})')
axes[2].imshow(crop_img_nont)
axes[2].set_title(f'Crop {crop_size}x{crop_size} - was_nonT ({rand_x}, {rand_y})')
plt.tight_layout()
plt.show()

In [ ]:
#her2 데이터셋 patch 이미지와 label 저장
save_path='../../data/precise_BC_cell_scoring/'
create_dir(save_path+'her2/patch_images/')
create_dir(save_path+'her2/labels/')
target_mpp=0.5
image_size=512
min_cells=10

for slide_idx in tqdm(range(len(her2_slide_list))):
    slide = openslide.OpenSlide(her2_slide_list[slide_idx])
    with open(her2_json_list[slide_idx], 'r') as f:
        data = json.load(f)
    
    # slide mpp 계산
    slide_mpp = float(slide.properties.get('openslide.mpp-x', 0.25))
    # level0에서 읽어야 할 patch 크기
    read_size = int(image_size * target_mpp / slide_mpp)
    
    slide_w, slide_h = slide.level_dimensions[0]
    
    # class 매핑
    class_map = {}
    for c in data['categories']:
        class_map[c['id']] = c['name']
    
    # class name -> YOLO class id
    class_to_id = {'class0': 0, 'class1': 1, 'class2': 2, 'class3': 3, 'other': 4}
    
    # annotation을 numpy array로 변환 (빠른 필터링)
    ann_list = []
    for ann in data['annotations']:
        x, y, w, h = ann['bbox']
        cat_name = class_map.get(ann['category_id'], 'other')
        if cat_name=='class4':
            cat_name='other'
        was_nonT = 1 if ann.get('was_nonT', False) else 0
        cls_id = class_to_id.get(cat_name, 4)
        ann_list.append([x, y, w, h, cls_id, was_nonT])
    
    if len(ann_list) == 0:
        continue
    ann_arr = np.array(ann_list)
    
    slide_name = her2_slide_list[slide_idx].split('/')[-1].replace(';', '_').replace(' ', '_')
    
    # grid 기반 패치 생성
    patch_count = 0
    for start_y in tqdm(range(0, slide_h - read_size + 1, read_size)):
        for start_x in range(0, slide_w - read_size + 1, read_size):
            # 해당 패치 범위 내 annotation 필터링
            mask = (ann_arr[:, 0] >= start_x) & (ann_arr[:, 0] < start_x + read_size) & \
                   (ann_arr[:, 1] >= start_y) & (ann_arr[:, 1] < start_y + read_size)
            patch_anns = ann_arr[mask]
            
            if len(patch_anns) < min_cells:
                continue
            
            # 패치 이미지 읽기 및 리사이즈
            patch_img = slide.read_region((start_x, start_y), 0, (read_size, read_size)).convert('RGB')
            patch_img = patch_img.resize((image_size, image_size), Image.LANCZOS)
            
            # JSON format label 생성 (normalized)
            scale = slide_mpp / target_mpp
            labels = []
            for ann in patch_anns:
                ax, ay, aw, ah, cls_id, was_nonT = ann
                # level0 좌표 → patch 내 좌표 → normalized
                cx = float(np.clip(((ax - start_x) + aw / 2) * scale / image_size, 0, 1))
                cy = float(np.clip(((ay - start_y) + ah / 2) * scale / image_size, 0, 1))
                nw = float(np.clip(aw * scale / image_size, 0, 1))
                nh = float(np.clip(ah * scale / image_size, 0, 1))
                labels.append({
                    "class_id": int(cls_id),
                    "cx": round(cx, 6),
                    "cy": round(cy, 6),
                    "w": round(nw, 6),
                    "h": round(nh, 6),
                    "was_nonT": int(was_nonT)
                })
            
            # 저장
            fname = f"{slide_name}_{start_x}_{start_y}"
            patch_img.save(os.path.join(save_path, 'her2/patch_images/', f"{fname}.png"))
            with open(os.path.join(save_path, 'her2/labels/', f"{fname}.json"), 'w') as f:
                json.dump(labels, f)
            patch_count += 1
    
    print(f"[{slide_idx}] {slide_name}: {patch_count} patches saved")
    slide.close()

print("Done!")

In [ ]:
#er/pr 데이터셋 patch 이미지와 label 저장
save_path='../../data/precise_BC_cell_scoring/'
create_dir(save_path+'er_pr/patch_images/')
create_dir(save_path+'er_pr/labels/')
target_mpp=0.5
image_size=512
min_cells=10

for slide_idx in tqdm(range(len(er_pr_slide_list))):
    slide = openslide.OpenSlide(er_pr_slide_list[slide_idx])
    with open(er_pr_json_list[slide_idx], 'r') as f:
        data = json.load(f)
    
    # slide mpp 계산
    slide_mpp = float(slide.properties.get('openslide.mpp-x', 0.25))
    # level0에서 읽어야 할 patch 크기
    read_size = int(image_size * target_mpp / slide_mpp)
    
    slide_w, slide_h = slide.level_dimensions[0]
    
    # class 매핑
    class_map = {}
    for c in data['categories']:
        class_map[c['id']] = c['name']
    
    # class name -> YOLO class id
    class_to_id = {'class0': 0, 'class1': 1, 'class2': 2, 'class3': 3, 'other': 4}
    
    # annotation을 numpy array로 변환 (빠른 필터링)
    ann_list = []
    for ann in data['annotations']:
        x, y, w, h = ann['bbox']
        cat_name = class_map.get(ann['category_id'], 'other')
        if cat_name=='non_tumor':
            cat_name='other'
        was_nonT = 1 if ann.get('was_nonT', False) else 0
        cls_id = class_to_id.get(cat_name, 4)
        ann_list.append([x, y, w, h, cls_id, was_nonT])
    
    if len(ann_list) == 0:
        continue
    ann_arr = np.array(ann_list)
    
    slide_name = er_pr_slide_list[slide_idx].split('/')[-1].replace(';', '_').replace(' ', '_')
    
    # grid 기반 패치 생성
    patch_count = 0
    for start_y in tqdm(range(0, slide_h - read_size + 1, read_size)):
        for start_x in range(0, slide_w - read_size + 1, read_size):
            # 해당 패치 범위 내 annotation 필터링
            mask = (ann_arr[:, 0] >= start_x) & (ann_arr[:, 0] < start_x + read_size) & \
                   (ann_arr[:, 1] >= start_y) & (ann_arr[:, 1] < start_y + read_size)
            patch_anns = ann_arr[mask]
            
            if len(patch_anns) < min_cells:
                continue
            
            # 패치 이미지 읽기 및 리사이즈
            patch_img = slide.read_region((start_x, start_y), 0, (read_size, read_size)).convert('RGB')
            patch_img = patch_img.resize((image_size, image_size), Image.LANCZOS)
            
            # JSON format label 생성 (normalized)
            scale = slide_mpp / target_mpp
            labels = []
            for ann in patch_anns:
                ax, ay, aw, ah, cls_id, was_nonT = ann
                # level0 좌표 → patch 내 좌표 → normalized
                cx = float(np.clip(((ax - start_x) + aw / 2) * scale / image_size, 0, 1))
                cy = float(np.clip(((ay - start_y) + ah / 2) * scale / image_size, 0, 1))
                nw = float(np.clip(aw * scale / image_size, 0, 1))
                nh = float(np.clip(ah * scale / image_size, 0, 1))
                labels.append({
                    "class_id": int(cls_id),
                    "cx": round(cx, 6),
                    "cy": round(cy, 6),
                    "w": round(nw, 6),
                    "h": round(nh, 6),
                    "was_nonT": int(was_nonT)
                })
            
            # 저장
            fname = f"{slide_name}_{start_x}_{start_y}"
            patch_img.save(os.path.join(save_path, 'er_pr/patch_images/', f"{fname}.png"))
            with open(os.path.join(save_path, 'er_pr/labels/', f"{fname}.json"), 'w') as f:
                json.dump(labels, f)
            patch_count += 1
    
    print(f"[{slide_idx}] {slide_name}: {patch_count} patches saved")
    slide.close()

print("Done!")